# updated_sample_hypothesis_testing

Formal statistical testing for H1/H2 on updated sample project.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kruskal, wilcoxon
from sklearn.ensemble import RandomForestRegressor, StackingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'raw' / 'yellow_taxi_representative_sample_2021_2023_distribution.csv'
OUT_RESULTS = ROOT / 'reports' / 'results'
OUT_FIG = ROOT / 'reports' / 'figures'
OUT_RESULTS.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)


def load_data(path):
    df = pd.read_csv(path)
    df['pickup_year'] = pd.to_numeric(df['pickup_year'], errors='coerce')
    df['pickup_month'] = pd.to_numeric(df['pickup_month'], errors='coerce')
    df['sample_rows'] = pd.to_numeric(df['sample_rows'], errors='coerce')
    df = df.dropna().copy()
    df['date'] = pd.to_datetime(df['pickup_year'].astype(int).astype(str) + '-' + df['pickup_month'].astype(int).astype(str).str.zfill(2) + '-01')
    df = df.sort_values('date').reset_index(drop=True)
    return df


def build_features(df):
    x = df.copy()
    x['year'] = x['date'].dt.year
    x['month'] = x['date'].dt.month
    x['quarter'] = x['date'].dt.quarter
    x['month_sin'] = np.sin(2*np.pi*x['month']/12)
    x['month_cos'] = np.cos(2*np.pi*x['month']/12)
    x['lag_1'] = x['sample_rows'].shift(1)
    x['lag_2'] = x['sample_rows'].shift(2)
    x['lag_3'] = x['sample_rows'].shift(3)
    x['roll_mean_3'] = x['sample_rows'].shift(1).rolling(3, min_periods=1).mean()
    x['roll_std_3'] = x['sample_rows'].shift(1).rolling(3, min_periods=1).std().fillna(0.0)
    return x.dropna().reset_index(drop=True)


def split_time(df):
    t = np.array(sorted(df['date'].unique()))
    n = len(t)
    tr_end = int(n*0.70)
    va_end = int(n*0.85)
    tr = df[df['date'].isin(set(t[:tr_end]))].copy()
    va = df[df['date'].isin(set(t[tr_end:va_end]))].copy()
    te = df[df['date'].isin(set(t[va_end:]))].copy()
    return tr, va, te


def metrics(y, p):
    return {
      'mae': float(mean_absolute_error(y,p)),
      'rmse': float(np.sqrt(mean_squared_error(y,p))),
      'r2': float(r2_score(y,p)),
      'residual_variance': float(np.var(y-p))
    }


In [2]:
df=load_data(DATA); feat=build_features(df); train,valid,test=split_time(feat)
features=['year','month','quarter','month_sin','month_cos','lag_1','lag_2','lag_3','roll_mean_3','roll_std_3']; target='sample_rows'
models={
  'boosting_xgboost': XGBRegressor(n_estimators=120,max_depth=4,learning_rate=0.08,objective='reg:squarederror',random_state=42,n_jobs=1),
  'bagging_random_forest': RandomForestRegressor(n_estimators=200,max_depth=8,random_state=42,n_jobs=1),
  'stacking_ensemble': StackingRegressor(estimators=[('xgb', XGBRegressor(n_estimators=80,max_depth=3,learning_rate=0.1,objective='reg:squarederror',random_state=42,n_jobs=1)),('rf',RandomForestRegressor(n_estimators=120,random_state=42,n_jobs=1)),('hgb',HistGradientBoostingRegressor(max_depth=6,random_state=42))], final_estimator=RidgeCV(alphas=np.logspace(-3,3,13)), passthrough=True, cv=3, n_jobs=1)
}
preds=pd.DataFrame(index=test.index); rows=[]
for n,m in models.items():
 m.fit(train[features],train[target]); p=m.predict(test[features]); preds[n]=p; rr=metrics(test[target].to_numpy(),p); rr['model']=n; rows.append(rr)
model_df=pd.DataFrame(rows).sort_values('mae').reset_index(drop=True)
model_df


,mae,rmse,r2,residual_variance,model
0,1.463367,1.517050,-13.384013,0.160000,boosting_xgboost
1,3.116000,3.139414,-60.599500,0.146464,bagging_random_forest
2,11.133362,11.140817,-774.736238,0.166038,stacking_ensemble


In [3]:
def boot_ci(a,b,n=2000,seed=42):
 rng=np.random.default_rng(seed); d=a-b; idx=rng.integers(0,len(d),size=(n,len(d))); m=d[idx].mean(axis=1); lo,hi=np.percentile(m,[2.5,97.5]); return float(d.mean()), float(lo), float(hi)

def perm_p(a,b,n=5000,seed=42):
 rng=np.random.default_rng(seed); d=a-b; obs=abs(d.mean()); s=rng.choice([-1,1],size=(n,len(d))); perm=np.abs((d*s).mean(axis=1)); return float((np.sum(perm>=obs)+1)/(n+1))

rows=[]
y=test[target].to_numpy()
cols=list(preds.columns)
for i in range(len(cols)):
 for j in range(i+1,len(cols)):
  a,b=cols[i],cols[j]
  ea=np.abs(y-preds[a].to_numpy()); eb=np.abs(y-preds[b].to_numpy())
  md,lo,hi=boot_ci(ea,eb,n=3000,seed=42)
  p_perm=perm_p(ea,eb,n=5000,seed=42)
  try: p_w=float(wilcoxon(ea,eb).pvalue)
  except Exception: p_w=float('nan')
  rows.append({'model_a':a,'model_b':b,'mean_abs_error_diff_a_minus_b':md,'ci95_low':lo,'ci95_high':hi,'p_permutation':p_perm,'p_wilcoxon':p_w})
pairwise_df=pd.DataFrame(rows)
pairwise_df.to_csv(OUT_RESULTS/'pairwise_stat_tests.csv', index=False)
pairwise_df


,model_a,model_b,mean_abs_error_diff_a_minus_b,ci95_low,ci95_high,p_permutation,p_wilcoxon
0,boosting_xgboost,bagging_random_forest,-1.652633,-1.676633,-1.628633,0.061988,0.0625
1,boosting_xgboost,stacking_ensemble,-9.669996,-10.135554,-8.858470,0.061988,0.0625
2,bagging_random_forest,stacking_ensemble,-8.017362,-8.482920,-7.229837,0.061988,0.0625


In [4]:
# H1 seasonality
month_groups=[g['sample_rows'].to_numpy() for _,g in feat.groupby('month') if len(g)>0]
if len(month_groups)>=3:
 stat=kruskal(*month_groups)
 h1={'tested':True,'p_value':float(stat.pvalue),'support':bool(stat.pvalue<0.05)}
else:
 h1={'tested':False}

# H2 ensemble better than boosting
boost=float(model_df.loc[model_df['model']=='boosting_xgboost','mae'].iloc[0])
bag=float(model_df.loc[model_df['model']=='bagging_random_forest','mae'].iloc[0])
stack=float(model_df.loc[model_df['model']=='stacking_ensemble','mae'].iloc[0])
subset=pairwise_df[((pairwise_df['model_a'].str.contains('ensemble|bagging')) & (pairwise_df['model_b']=='boosting_xgboost')) | ((pairwise_df['model_b'].str.contains('ensemble|bagging')) & (pairwise_df['model_a']=='boosting_xgboost'))]
sig=False
for _,r in subset.iterrows():
 lo,hi,p=r['ci95_low'],r['ci95_high'],r['p_permutation']
 if r['model_a'] in ['bagging_random_forest','stacking_ensemble']:
  direction_ok = hi < 0
 else:
  direction_ok = lo > 0
 sig = sig or (p<0.05 and direction_ok)

h2={'tested':True,'support':bool(((bag<boost) or (stack<boost)) and sig),'boosting_mae':boost,'bagging_mae':bag,'stacking_mae':stack}
summary={'best_model':str(model_df.iloc[0]['model']),'hypothesis_tests':{'h1_seasonality':h1,'h2_ensemble_better_than_boosting':h2}}
with open(OUT_RESULTS/'hypothesis_summary.json','w',encoding='utf-8') as f: json.dump(summary,f,indent=2)
summary


{'best_model': 'boosting_xgboost',
 'hypothesis_tests': {'h1_seasonality': {'tested': True,
   'p_value': 0.9520819621960708,
   'support': False},
  'h2_ensemble_better_than_boosting': {'tested': True,
   'support': False,
   'boosting_mae': 1.4633667469024658,
   'bagging_mae': 3.115999999999997,
   'stacking_mae': 11.13336245804312}}}

In [5]:
pd.DataFrame([{'stage':'data_loading','status':'done'},{'stage':'feature_engineering','status':'done'},{'stage':'temporal_split_train_validation_test','status':'done'},{'stage':'modeling','status':'done'},{'stage':'statistical_testing','status':'done'},{'stage':'reporting','status':'done'}]).to_csv(OUT_RESULTS/'roadmap_execution.csv',index=False)
print('saved:', OUT_RESULTS)


saved: /Users/heliamahmoodzadeh/Documents/GitHub/nyc-mobility-analysis/updated_GITHUB_sample_code/reports/results
